# Data Leakage

In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold,
    GroupKFold, TimeSeriesSplit, GridSearchCV
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PowerTransformer
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, mutual_info_classif, RFE
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.dummy import DummyClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

X_arr, y = make_classification(
    n_samples=1200, n_features=10, n_informative=6,
    n_redundant=2, random_state=RANDOM_STATE
)
df = pd.DataFrame(X_arr, columns=[f"feature_{i}" for i in range(10)])
df["target"] = y
df.head()


D:\Users\cccc\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,target
0,-0.407747,-1.005971,-0.907327,0.867376,-1.892688,-0.034942,-0.992483,-1.739297,-0.727128,1.021992,1
1,-0.876189,0.109140,1.584494,0.815768,-2.419784,0.693706,-0.171851,-0.252788,-2.030066,0.397911,0
2,-0.142072,-0.273983,1.637082,0.660069,2.285809,-0.630458,-0.383266,1.163070,0.406553,-0.303175,1
3,-1.293832,0.278527,-0.078788,2.039231,2.410870,-0.766353,2.292495,-2.499796,-0.861652,-4.788024,1
4,-0.776777,-3.472437,-1.482397,3.625974,0.760436,1.478121,-2.370914,-3.097145,-0.631403,1.012133,0


## 1. Target leakage

In [2]:
target_leak_df = df.copy()
target_leak_df["post_event_status"] = (
    target_leak_df["target"] * 0.95
    + np.random.normal(0, 0.05, len(target_leak_df))
)

corr = target_leak_df.corr(numeric_only=True)["target"].abs().sort_values(ascending=False)
corr.head()


target               1.000000
post_event_status    0.994626
feature_1            0.519262
feature_4            0.373472
feature_9            0.341069
Name: target, dtype: float64

In [3]:
X = target_leak_df.drop(columns="target")
y = target_leak_df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)

print("Accuracy with leaky feature:", accuracy_score(y_test, rf.predict(X_test)))

pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head()


Accuracy with leaky feature: 1.0


post_event_status    0.714765
feature_1            0.082329
feature_4            0.058271
feature_9            0.045159
feature_0            0.023640
dtype: float64

In [4]:
# Remove only after confirming the feature is unavailable at prediction time
forbidden = ["post_event_status", "future_price", "recovery_amount", "final_exam_score"]

X_clean = target_leak_df.drop(
    columns=["target"] + [c for c in forbidden if c in target_leak_df.columns]
)
y_clean = target_leak_df["target"]


## 2. Preprocessing leakage

In [5]:
X = df.drop(columns="target")
y = df["target"]

# WRONG pattern (for illustration only)
scaler_bad = StandardScaler()
X_bad = scaler_bad.fit_transform(X)  # sees all rows

# CORRECT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("TRAIN -> fit + transform")
print("TEST  -> transform only")


TRAIN -> fit + transform
TEST  -> transform only


## 3. Feature-selection leakage

Mutual Information, ANOVA, Chi-square, RFE and model-based selectors must not see validation/test targets before evaluation.

Correct approach: put feature selection **inside the pipeline**.


In [6]:
X = df.drop(columns="target")
y = df["target"]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

mi_pipe = Pipeline([
    ("selector", SelectKBest(mutual_info_classif, k=5)),
    ("model", LogisticRegression(max_iter=2000))
])

scores = cross_val_score(mi_pipe, X, y, cv=cv, scoring="accuracy")
print("Leakage-safe MI feature selection:", scores.mean())


Leakage-safe MI feature selection: 0.8066666666666666


## 4. Cross-validation leakage

CV itself does **not** prevent leakage.

Bad:
```python
X = scaler.fit_transform(X)
X = selector.fit_transform(X, y)
cross_val_score(model, X, y)
```

Safe rule: every data-dependent step belongs inside the pipeline.


In [8]:
safe_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("selector", SelectKBest(mutual_info_classif, k=6)),
    ("model", LogisticRegression(max_iter=2000))
])

scores = cross_val_score(
    safe_pipe,
    df.drop(columns="target"),
    df["target"],
    cv=cv,
    scoring="accuracy"
)

print("Full leakage-safe CV:", scores.mean())


Full leakage-safe CV: 0.7991666666666667


## 5. Hyperparameter

If you repeatedly look at test performance to choose:
- model family
- features
- preprocessing
- hyperparameters
- classification threshold

the test set is no longer truly unseen.

Use CV/GridSearch on training data, then evaluate the final test set once.


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns="target"), df["target"],
    test_size=0.2, stratify=df["target"], random_state=RANDOM_STATE
)

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000))
])

grid = GridSearchCV(
    pipe,
    {"model__C": [0.01, 0.1, 1, 10]},
    cv=5,
    scoring="accuracy"
)

grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Training CV score:", grid.best_score_)
print("Untouched test accuracy:", accuracy_score(y_test, grid.predict(X_test)))


Best params: {'model__C': 0.01}
Training CV score: 0.8052083333333332
Untouched test accuracy: 0.8


## 6. Duplicate leakage

Exact or near-duplicate rows can cross train/test boundaries. This is common in scraped, image, text, and augmented datasets.


In [10]:
dup_df = pd.concat([df, df.iloc[:20]], ignore_index=True)

print("Duplicate rows:", dup_df.duplicated().sum())

# Remove only if they are truly accidental duplicates
dedup_df = dup_df.drop_duplicates()

print("Before:", len(dup_df))
print("After :", len(dedup_df))


Duplicate rows: 20
Before: 1220
After : 1200


In [11]:
def exact_row_overlap(X_train, X_test):
    train_rows = set(map(tuple, X_train.to_numpy()))
    test_rows = set(map(tuple, X_test.to_numpy()))
    return len(train_rows & test_rows)

X = dedup_df.drop(columns="target")
y = dedup_df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print("Exact train-test overlap:", exact_row_overlap(X_train, X_test))


Exact train-test overlap: 0


## 7. Time-series / future leakage

For forecasting:
- preserve chronology
- do not randomly shuffle
- do not use future rows to predict past rows
- use `TimeSeriesSplit` or walk-forward validation


In [12]:
n = 400
time_df = pd.DataFrame({
    "date": pd.date_range("2024-01-01", periods=n, freq="D"),
    "price": 100 + np.cumsum(np.random.normal(0, 1, n))
})

time_df["return"] = time_df["price"].pct_change()
time_df["target"] = (time_df["price"].shift(-1) > time_df["price"]).astype(int)
time_df = time_df.dropna().reset_index(drop=True)

split = int(len(time_df) * 0.8)
train_ts = time_df.iloc[:split]
test_ts = time_df.iloc[split:]

print("Train max:", train_ts["date"].max())
print("Test min :", test_ts["date"].min())

assert train_ts["date"].max() < test_ts["date"].min()


Train max: 2024-11-15 00:00:00
Test min : 2024-11-16 00:00:00


In [13]:
tscv = TimeSeriesSplit(n_splits=5)
X_ts = time_df[["price", "return"]]
y_ts = time_df["target"]

for i, (tr, va) in enumerate(tscv.split(X_ts), 1):
    print(f"Fold {i}: train {tr[0]}..{tr[-1]} | validation {va[0]}..{va[-1]}")


Fold 1: train 0..68 | validation 69..134
Fold 2: train 0..134 | validation 135..200
Fold 3: train 0..200 | validation 201..266
Fold 4: train 0..266 | validation 267..332
Fold 5: train 0..332 | validation 333..398


# 8. Complete leakage-safe mixed-data workflow

The pattern below is what you should aim for in real tabular ML:
1. define `X` and `y`
2. create untouched test split
3. put learnable preprocessing in a pipeline
4. run CV only on training data
5. tune only on training data
6. fit final pipeline
7. evaluate final test set once


In [14]:
mixed_df = pd.DataFrame({
    "age": np.random.randint(18, 65, 1000),
    "income": np.random.normal(60000, 15000, 1000),
    "city": np.random.choice(["Lahore", "Karachi", "Islamabad"], 1000),
    "occupation": np.random.choice(["Engineer", "Teacher", "Student", "Other"], 1000),
    "target": np.random.binomial(1, 0.45, 1000)
})

mixed_df.loc[np.random.rand(len(mixed_df)) < 0.05, "income"] = np.nan
mixed_df.loc[np.random.rand(len(mixed_df)) < 0.05, "city"] = None

X = mixed_df.drop(columns="target")
y = mixed_df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

num_cols = ["age", "income"]
cat_cols = ["city", "occupation"]

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])

final_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=2000))
])

cv_scores = cross_val_score(
    final_pipe,
    X_train,
    y_train,
    cv=5,
    scoring="accuracy"
)

print("Training CV mean:", cv_scores.mean())

final_pipe.fit(X_train, y_train)
test_pred = final_pipe.predict(X_test)

print("Final untouched test accuracy:", accuracy_score(y_test, test_pred))


Training CV mean: 0.5225
Final untouched test accuracy: 0.555


## Note: Three questions that catch most leakage

1. **Feature availability:** Would I know this feature at the exact moment of prediction?
2. **Data separation:** Did validation/test data help preprocess, select, resample, or tune the model?
3. **Independence:** Are train and validation/test truly independent in terms of people/groups, time, duplicates, and derived samples?
